# E19c: OMNI-DC on ARKitScenes, seeds 3-9 forward + E19 seeds 0-9 (confirmatory)

**Before you start**
1. Upload `omni_code.tgz` (from `CV_Project/colab/`) to `MyDrive/spacal/omni_code.tgz`.
2. Runtime -> Change runtime type -> **A100 GPU**, **High-RAM** on.
3. Run the cells top to bottom and keep this tab open. Every long step runs in the foreground, so Colab does not see the runtime as idle.

All outputs go to Drive as they are produced: forward chunks to `MyDrive/spacal/omnidc_raw/c39/` (112 files when complete), results to `MyDrive/spacal/results_omnidc/e19_conf_s0..s9/`.
If the runtime dies, start again from cell 1: finished chunks and finished seeds are skipped.

Expected time: about 10 min setup, up to about 2.3 h forward (less if chunks survived), about 45 min E19 on CPU.
When cell 7 prints `ALL DONE`, use Runtime -> Disconnect and delete runtime.

**To save compute units:** after cell 5 finishes, disconnect the A100 runtime, switch to a **CPU (High-RAM)** runtime, and run cells 1, 2, 6, 7. The E19 step needs no GPU.

Bundle check: cell 2 prints the md5 of `omni_code.tgz`; it should be `1004a1f7f8b7b9ce870e2767e2b9cce3`.

In [ ]:
# 1. Drive + GPU check
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
D = '/content/drive/MyDrive/spacal'
assert os.path.exists(f'{D}/omni_code.tgz'), 'upload omni_code.tgz to MyDrive/spacal/ first'
assert os.path.exists(f'{D}/omnidc_merged') and os.path.exists(f'{D}/prepared_new/arkitscenes__unidepthv2-large.npz')
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode: raise RuntimeError(f'failed ({r.returncode}): {cmd[:80]}')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2')

In [ ]:
# 2. Code, OMNI-DC repo, weights (~3 min)
os.environ.update(SPACAL_DATA_ROOT='/content/data', HF_HOME='/content/hf_cache', PYTHONUNBUFFERED='1',
                  PYTHONPATH='/content/CV_Project', MPLCONFIGDIR='/content/mpl')
A, DA = '/content/omni_anchors/arkitscenes', '/content/data/arkitscenes'
RAW, DRAW, MERGED = '/content/omnidc_raw/c39', f'{D}/omnidc_raw/c39', '/content/omnidc_merged'
sh(f'cd /content && rm -rf CV_Project omni_anchors && tar xzf {D}/omni_code.tgz -C /content && md5sum {D}/omni_code.tgz')
sh('cd /content && ([ -d OMNI-DC ] || git clone -q --depth 1 https://github.com/princeton-vl/OMNI-DC)')
sh('pip install -q timm einops huggingface_hub safetensors')
sh("""python -c "from huggingface_hub import hf_hub_download as d; print(d('zuoym15/OMNI-DC', 'model.safetensors'))" """)

In [ ]:
# 3. ARKitScenes validation frames (~5-10 min, 8 parallel downloads)
sh(f'python /content/CV_Project/scripts/fetch_arkitscenes.py --dst {DA} --limit 1')
sh(' & '.join(f'python /content/CV_Project/scripts/fetch_arkitscenes.py --dst {DA} --shard {s} --n-shards 8 --tmp /tmp/arkit_dl{s} > /content/fetch_{s}.log 2>&1' for s in range(8)) + ' & wait')
sh(f'tail -n 1 /content/fetch_*.log; du -sh {DA}')

In [ ]:
# 4. OMNI-DC forward, laser anchors, seeds 3-9 x {uniform, cluster} (up to ~2.3 h). Syncs to Drive every minute.
import time, glob
os.makedirs(RAW, exist_ok=True); os.makedirs(DRAW, exist_ok=True)
sh(f"rsync -a --exclude='*.tmp.npz' {DRAW}/ {RAW}/")
n_chunks = lambda d: len([f for f in glob.glob(f'{d}/*.npz') if not f.endswith('.tmp.npz')])
print('restored chunks:', n_chunks(RAW))
masks = [f'{p}/k200_seed{s}_draw0' for p in ('uniform', 'cluster') for s in range(3, 10)]
procs = [subprocess.Popen(['python', '-m', 'scripts.omnidc_forward', '--omnidc', '/content/OMNI-DC', '--anchors', A,
                           '--masks', *masks, '--sources', 'clean', '--out', RAW, '--shard', str(s), '--n-shards', '2',
                           '--workers', '4'], cwd='/content/CV_Project',
                          stdout=open(f'/content/forward_s{s}.log', 'w'), stderr=subprocess.STDOUT) for s in (0, 1)]
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    subprocess.run(f"rsync -a --exclude='*.tmp.npz' {RAW}/ {DRAW}/", shell=True)
    print(f'{(time.time() - t0) / 60:5.0f} min  chunks local {n_chunks(RAW)}  on Drive {n_chunks(DRAW)}', flush=True)
subprocess.run(f"rsync -a --exclude='*.tmp.npz' {RAW}/ {DRAW}/", shell=True)
codes = [p.returncode for p in procs]
print('exit codes', codes, ' chunks on Drive', n_chunks(DRAW), '(expect 112)')
if any(codes): sh('tail -n 30 /content/forward_s0.log /content/forward_s1.log')

In [ ]:
# 5. Merge seeds 3-9 (CPU, ~5 min) and save merged files to Drive
os.makedirs(MERGED, exist_ok=True)
sh(f'cd /content/CV_Project && python -m scripts.omnidc_merge --anchors {A} --pred-dir {RAW} --out {MERGED}')
sh(f'rsync -a {MERGED}/ {D}/omnidc_merged/ && ls {D}/omnidc_merged | grep -c clean')

In [ ]:
# 6. Inputs for E19: frozen-model table + merged OMNI-DC seeds 0-9 from Drive; prediction diagnostic
#    (after a restart on a CPU runtime, cells 1, 2, 6, 7 are enough once cell 5 has finished once)
import glob
os.makedirs('/content/prepared', exist_ok=True); os.makedirs(MERGED, exist_ok=True)
sh(f'cp {D}/prepared_new/arkitscenes__unidepthv2-large.npz /content/prepared/')
sh(' && '.join(f'cp -n {D}/omnidc_merged/{p}__k200_seed{s}_draw0__clean.npz {MERGED}/' for p in ('uniform', 'cluster') for s in range(10)))
sh(f'ls {MERGED}')
import numpy as np
gt = np.load(f'{A}/table_values.npz')['gt'].astype(np.float64)
for f in sorted(glob.glob(f'{MERGED}/*__clean.npz')):
    pr = np.load(f)['pred'].astype(np.float64); ok = np.isfinite(pr) & (gt > 0)
    le = np.abs(np.log(np.maximum(pr[ok], 1e-3) / gt[ok]))
    print(f'{os.path.basename(f):42s} pred [{pr[ok].min():.3g}, {pr[ok].max():.3g}]  |log err| q99 {np.quantile(le, .99):.3f} max {le.max():.3f}')

In [ ]:
# 7. E19 for seeds 0-9 (CPU, 3 at a time, ~45 min). Results go straight to Drive; finished seeds are skipped.
import time
env = dict(os.environ, SPACAL_RESULTS_ROOT=f'{D}/results_omnidc', SPACAL_PREPARED_ROOT='/content/prepared', OMP_NUM_THREADS='3')
def done(s): return bool(glob.glob(f'{D}/results_omnidc/e19_conf_s{s}/**/summary.csv', recursive=True))
todo = [s for s in range(10) if not done(s)]
print('seeds to run:', todo)
os.makedirs(f'{D}/logs/e19c', exist_ok=True)
for i in range(0, len(todo), 3):
    batch = todo[i:i + 3]
    ps = {s: subprocess.Popen(['python', '-m', 'experiments.e19_omnidc', '--exp-id', f'e19_conf_s{s}', '--anchors', A,
                               '--omni', MERGED, '--patterns', 'uniform', 'cluster', '--seeds', str(s), '--draws', '0',
                               '--omni-sources', 'clean'], cwd='/content/CV_Project', env=env,
                              stdout=open(f'{D}/logs/e19c/e19c_s{s}.log', 'w'), stderr=subprocess.STDOUT) for s in batch}
    for s, p in ps.items():
        p.wait(); print(f'seed {s}: exit {p.returncode}  {time.strftime("%H:%M:%S")}', flush=True)
missing = [s for s in range(10) if not done(s)]
print('ALL DONE' if not missing else f'MISSING seeds {missing}: see MyDrive/spacal/logs/e19c/e19c_s<seed>.log, then rerun this cell')

In [ ]:
# 8. Archive the E19c results on Drive + AbsRel of OMNI-DC vs UniDepthV2 on test pixels OMNI-DC did not see (~3 min)
import sys, glob, tarfile, shutil, numpy as np, pandas as pd
sys.path.insert(0, '/content/CV_Project'); os.environ['SPACAL_PREPARED_ROOT'] = '/content/prepared'
os.makedirs('/content/prepared', exist_ok=True)
if not os.path.exists('/content/prepared/arkitscenes__unidepthv2-large.npz'):
    shutil.copy(f'{D}/prepared_new/arkitscenes__unidepthv2-large.npz', '/content/prepared/')
with tarfile.open(f'{D}/e19c_results.tgz', 'w:gz') as t:
    for f in sorted(glob.glob(f'{D}/results_omnidc/e19_conf_s*/**/*', recursive=True)):
        if f.endswith(('.csv', '.json', '.md')):
            t.add(f, arcname=os.path.relpath(f, f'{D}/results_omnidc'))
print('archive:', os.path.getsize(f'{D}/e19c_results.tgz') // 1024, 'KB')

from experiments.common import load_table, split_tables
from src.data.protocols import PROTOCOLS, clamp_pred
base = load_table('arkitscenes', 'unidepthv2-large'); prot = PROTOCOLS['arkitscenes']
img, gt = base.image_of_pixel(), base.pix['gt'].astype(np.float64)
uni = clamp_pred(base.pix['pred'], prot).astype(np.float64)
rows = []
for p in ('uniform', 'cluster'):
    for s in range(10):
        stem = f'k200_seed{s}_draw0'
        with np.load(f'{A}/{p}/{stem}.npz') as z: seen = z['structured']
        with np.load(f'{MERGED}/{p}__{stem}__clean.npz') as z: om = z['pred']
        ok_img = np.bincount(img, weights=~np.isfinite(om), minlength=base.n_images) == 0
        test = split_tables(base.subset(np.flatnonzero(ok_img)), s)[2]
        m = np.isin(base.img['sample_id'], test.img['sample_id'])[img] & ~seen & (gt > 0)
        om_c = clamp_pred(np.where(np.isfinite(om), om, 1.0), prot).astype(np.float64)
        for name, pr in (('omnidc', om_c), ('unidepth', uni)):
            r = np.abs(pr[m] - gt[m]) / gt[m]
            d1 = np.maximum(pr[m] / gt[m], gt[m] / pr[m]) < 1.25
            n = np.bincount(img[m], minlength=base.n_images)
            per_img = np.bincount(img[m], weights=r, minlength=base.n_images)[n > 0] / n[n > 0]
            rows.append({'pattern': p, 'seed': s, 'centre': name, 'n_test_img': int((n > 0).sum()),
                         'absrel_pix': r.mean(), 'absrel_img': per_img.mean(), 'delta1': d1.mean()})
df = pd.DataFrame(rows); df.to_csv(f'{D}/e19c_absrel.csv', index=False)
print(df.groupby(['pattern', 'centre'])[['absrel_pix', 'absrel_img', 'delta1']].mean().round(4))
from google.colab import files
files.download(f'{D}/e19c_results.tgz'); files.download(f'{D}/e19c_absrel.csv')